# Bài 2.9 · Giảm chiều trên ảnh chữ số thật

**Câu hỏi của buổi này:** một ảnh chữ số 28 × 28 là một điểm trong không gian 784 chiều. Cần bao nhiêu chiều để giữ gần hết thông tin, và bản đồ 2 chiều nào cho nhìn, bản đồ nào cho đo?

Dữ liệu là **MNIST**: chữ số viết tay 0–9. Lấy 10 000 ảnh train và 2 000 ảnh test.

| Mục | Sẽ thấy |
|---|---|
| 1. Dữ liệu | bảng hai tập, ảnh mẫu |
| 2. PCA: phương sai giữ được | đường phương sai tích lũy, số chiều cho 80/90/95% |
| 3. Thành phần chính trông thế nào | ảnh trung bình và 12 thành phần đầu |
| 4. Tái tạo | một chữ số dựng lại từ 2, 10, 50, 150 chiều |
| 5. Bản đồ 2 chiều | PCA so với t-SNE ở ba perplexity |
| 6. Bản đồ nào giữ khoảng cách | tương quan khoảng cách gốc và khoảng cách trên bản đồ |
| 7. PCA làm tiền xử lý | accuracy và thời gian theo số chiều giữ lại |
| 8. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **Internet** trong *Session options*; CPU là đủ, chạy hết khoảng 3 phút. Bài giảng: [ai-canuc80k · bài 2.9](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/2.9).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, sklearn, torchvision
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
N_TRAIN, N_TEST, N_MAP = 10000, 2000, 3000
table(pd.DataFrame([('scikit-learn', sklearn.__version__), ('numpy', np.__version__)], columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. Dữ liệu

Mỗi ảnh được duỗi thành một vector 784 số trong [0, 1].

In [ ]:
with quiet():
    tr = torchvision.datasets.MNIST('/tmp/mnist', train=True, download=True)
    te = torchvision.datasets.MNIST('/tmp/mnist', train=False, download=True)
rng = np.random.default_rng(0)
i_tr, i_te = rng.permutation(60000)[:N_TRAIN], rng.permutation(10000)[:N_TEST]
X = tr.data.numpy()[i_tr].reshape(-1, 784) / 255.0; y = tr.targets.numpy()[i_tr]
Xte = te.data.numpy()[i_te].reshape(-1, 784) / 255.0; yte = te.targets.numpy()[i_te]
table(pd.DataFrame([('train', X.shape[0], X.shape[1]), ('test', Xte.shape[0], Xte.shape[1])], columns=['tập', 'số ảnh', 'số chiều']), 'MNIST')
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for k in range(10):
    for r in range(2):
        axes[r, k].imshow(X[np.flatnonzero(y == k)[r]].reshape(28, 28), cmap='gray_r'); axes[r, k].axis('off')
plt.tight_layout(); plt.show()

## 2. PCA: phương sai giữ được

PCA tìm những hướng mà dữ liệu trải rộng nhất. Chiếu lên $k$ hướng đầu giữ được một tỉ lệ phương sai; đường tích lũy cho biết cần bao nhiêu chiều. PCA được tính bằng SVD của ma trận dữ liệu đã trừ trung bình.

In [ ]:
pca = PCA(random_state=0).fit(X)
cum = np.cumsum(pca.explained_variance_ratio_)
need = {p: int(np.searchsorted(cum, p) + 1) for p in [0.8, 0.9, 0.95, 0.99]}
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(np.arange(1, 785), cum, color=C['blue'], lw=2)
for (p, k), c in zip(need.items(), [C['sand'], C['clay'], C['green'], C['gray']]):
    ax.scatter([k], [p], color=c, zorder=3); ax.annotate(f'{p:.0%}: {k} chiều', (k, p), xytext=(8, -14), textcoords='offset points', fontsize=10, color=c)
ax.set_xscale('log'); ax.set(xlabel='số thành phần giữ lại (thang log)', ylabel='phương sai tích lũy', title='784 chiều, nhưng phần lớn thông tin nằm trong vài chục chiều đầu')
plt.tight_layout(); plt.show()
table(pd.DataFrame([(f'{p:.0%}', k, k / 784) for p, k in need.items()], columns=['phương sai muốn giữ', 'số chiều cần', 'tỉ lệ so với 784']), 'Bao nhiêu chiều là đủ', fmt={'tỉ lệ so với 784': '{:.1%}'})

Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
check('SVD tự tính ra đúng 5 hướng đầu của sklearn (sai khác dấu)', np.allclose(np.abs(Vt[:5]), np.abs(pca.components_[:5]), atol=1e-6))
check('phương sai giải thích = S² / (n − 1)', np.allclose(S[:5] ** 2 / (len(X) - 1), pca.explained_variance_[:5]))

## 3. Thành phần chính trông thế nào

Mỗi thành phần là một vector 784 số, tức cũng vẽ được thành ảnh 28 × 28. Đỏ là tăng độ đậm, xanh là giảm. Thành phần đầu thường tách "nét tròn to" với "nét đứng mảnh".

In [ ]:
fig, axes = plt.subplots(1, 13, figsize=(16, 1.9))
axes[0].imshow(pca.mean_.reshape(28, 28), cmap='gray_r'); axes[0].set_title('trung bình', fontsize=9)
for k in range(12):
    v = pca.components_[k].reshape(28, 28); m = np.abs(v).max()
    axes[k + 1].imshow(v, cmap='RdBu_r', vmin=-m, vmax=m); axes[k + 1].set_title(f'PC{k + 1}\n{pca.explained_variance_ratio_[k]:.1%}', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

z = pca.transform(X)[:, :2]
fig, axes = plt.subplots(1, 2, figsize=(13, 2.6))
for ax, j in zip(axes, [0, 1]):
    order = np.argsort(z[:, j]); picks = order[np.linspace(0, len(order) - 1, 9).astype(int)]
    ax.imshow(np.hstack([X[i].reshape(28, 28) for i in picks]), cmap='gray_r'); ax.axis('off')
    ax.set_title(f'xếp ảnh theo tọa độ PC{j + 1}: thấp → cao', fontsize=10)
plt.tight_layout(); plt.show()

## 4. Tái tạo

Chiếu một ảnh xuống $k$ chiều rồi dựng ngược về 784 chiều: $\hat x = \mu + \sum_{j \le k} (v_j^\top (x-\mu))\, v_j$. Ảnh dựng lại càng rõ khi $k$ càng lớn.

In [ ]:
ks = [2, 10, 30, 50, 150, 784]
idx = [int(np.flatnonzero(yte == d)[0]) for d in [2, 5, 8]]
fig, axes = plt.subplots(len(idx), len(ks) + 1, figsize=(12, 5.4))
err = []
for r, i in enumerate(idx):
    axes[r, 0].imshow(Xte[i].reshape(28, 28), cmap='gray_r'); axes[r, 0].set_title('gốc' if r == 0 else '', fontsize=10)
    for c, k in enumerate(ks, start=1):
        rec = pca.mean_ + (Xte[i] - pca.mean_) @ pca.components_[:k].T @ pca.components_[:k]
        axes[r, c].imshow(rec.reshape(28, 28), cmap='gray_r', vmin=0, vmax=1)
        if r == 0: axes[r, c].set_title(f'{k} chiều', fontsize=10)
for ax in axes.flat: ax.axis('off')
plt.tight_layout(); plt.show()
for k in ks:
    Z = (Xte - pca.mean_) @ pca.components_[:k].T
    err.append((k, np.mean((pca.mean_ + Z @ pca.components_[:k] - Xte) ** 2), cum[k - 1]))
rec_df = pd.DataFrame(err, columns=['số chiều', 'sai số bình phương trung bình (test)', 'phương sai giữ (train)'])
table(rec_df, 'Tái tạo 2 000 ảnh test', fmt={'sai số bình phương trung bình (test)': '{:.4f}', 'phương sai giữ (train)': '{:.1%}'})
check('giữ đủ 784 chiều thì dựng lại gần đúng tuyệt đối', rec_df.iloc[-1, 1] < 1e-10)
note(f"50 chiều (6% số chiều gốc) đã giữ {cum[49]:.0%} phương sai: chữ số dựng lại đọc được dễ dàng.")

## 5. Bản đồ 2 chiều: PCA và t-SNE

Lấy 3 000 ảnh, vẽ lên mặt phẳng, màu là nhãn thật (nhãn **không** được dùng khi tính bản đồ). PCA là phép chiếu tuyến tính; **t-SNE** đặt các điểm sao cho hàng xóm gần trong 784 chiều vẫn là hàng xóm gần trên bản đồ. Tham số **perplexity** của t-SNE xấp xỉ số hàng xóm mỗi điểm quan tâm.

In [ ]:
Xm, ym = X[:N_MAP], y[:N_MAP]
X50 = PCA(50, random_state=0).fit_transform(Xm)          # t-SNE thường chạy trên PCA 50 chiều cho nhanh
maps, tms = {'PCA': PCA(2, random_state=0).fit_transform(Xm)}, {}
for p in [5, 30, 100]:
    t = time.time(); maps[f't-SNE, perplexity {p}'] = TSNE(2, perplexity=p, init='pca', random_state=0).fit_transform(X50); tms[p] = time.time() - t
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (n, M) in zip(axes, maps.items()):
    sc = ax.scatter(M[:, 0], M[:, 1], c=ym, cmap='tab10', s=3); ax.set_title(n); ax.set_xticks([]); ax.set_yticks([])
    for d in range(10):
        cx, cy = np.median(M[ym == d], axis=0); ax.text(cx, cy, str(d), fontsize=13, weight='bold', ha='center', va='center', bbox=dict(fc='white', ec='none', alpha=.7, pad=1))
plt.tight_layout(); plt.show()
note(f"PCA chồng nhiều chữ số lên nhau; t-SNE tách phần lớn chữ số thành cụm riêng mà không biết nhãn, chỉ những chữ số viết giống nhau (như 4 và 9) còn sát nhau. Đổi perplexity thì hình đổi theo (t-SNE mất {tms[30]:.0f} giây với perplexity 30).")

## 6. Bản đồ nào giữ khoảng cách

t-SNE đẹp để **nhìn**, nhưng không nên dùng để **đo**: khoảng cách giữa hai cụm, kích thước một cụm trên bản đồ t-SNE không phản ánh không gian gốc. Đo bằng tương quan thứ hạng (Spearman) giữa khoảng cách mọi cặp điểm trong 784 chiều và trên bản đồ; và tỉ lệ 10 hàng xóm gần nhất được giữ lại.

In [ ]:
sub = np.random.default_rng(1).choice(N_MAP, 800, replace=False)
d0 = pdist(Xm[sub])
def knn_keep(A, B, k=10):
    from sklearn.neighbors import NearestNeighbors
    na = NearestNeighbors(n_neighbors=k + 1).fit(A).kneighbors(A, return_distance=False)[:, 1:]
    nb = NearestNeighbors(n_neighbors=k + 1).fit(B).kneighbors(B, return_distance=False)[:, 1:]
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(na, nb)])
rows = [(n, spearmanr(d0, pdist(M[sub])).correlation, knn_keep(Xm, M)) for n, M in maps.items()]
keep = pd.DataFrame(rows, columns=['bản đồ', 'tương quan khoảng cách (Spearman)', 'giữ 10 hàng xóm gần nhất'])
table(keep, 'Bản đồ giữ được gì của không gian 784 chiều', fmt={'tương quan khoảng cách (Spearman)': '{:.3f}', 'giữ 10 hàng xóm gần nhất': '{:.1%}'})
fig, ax = plt.subplots(figsize=(9, 3.2))
xs = np.arange(len(keep))
ax.bar(xs - .2, keep.iloc[:, 1], .4, color=C['blue'], label='tương quan khoảng cách'); ax.bar(xs + .2, keep.iloc[:, 2], .4, color=C['clay'], label='giữ hàng xóm gần')
ax.set_xticks(xs, keep['bản đồ'], fontsize=9); ax.set_ylim(0, 1.1); ax.legend(loc='upper left'); bar_labels(ax, '{:.2f}')
ax.set_title('PCA giữ khoảng cách xa; t-SNE giữ hàng xóm gần')
plt.tight_layout(); plt.show()
note(f"PCA giữ tương quan khoảng cách <b>{keep.iloc[0, 1]:.2f}</b> nhưng chỉ giữ {keep.iloc[0, 2]:.0%} hàng xóm gần; t-SNE (perplexity 30) ngược lại: {keep.iloc[2, 1]:.2f} và <b>{keep.iloc[2, 2]:.0%}</b>.")

## 7. PCA làm tiền xử lý

Giữ $k$ chiều đầu rồi đưa vào logistic regression. PCA được **fit trên train** trong cùng một Pipeline (bài 1.4), rồi áp nguyên cho test.

In [ ]:
rows = []
for k in [5, 10, 20, 50, 100, 200, None]:
    t = time.time()
    steps = [PCA(k, random_state=0)] if k else []
    m = make_pipeline(StandardScaler(with_std=False), *steps, LogisticRegression(max_iter=2000)).fit(X, y)
    rows.append(('784 (không PCA)' if k is None else str(k), (m.predict(Xte) == yte).mean(), time.time() - t))
pre = pd.DataFrame(rows, columns=['số chiều giữ lại', 'accuracy test', 'giây (fit)'])
table(pre, 'Logistic regression sau PCA', fmt={'accuracy test': '{:.3f}', 'giây (fit)': '{:.1f}'}, best='accuracy test')
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(pre['số chiều giữ lại'], pre['accuracy test'], marker='o', color=C['blue'], lw=2); ax.set_ylim(0.6, 1)
ax.set(xlabel='số chiều giữ lại', ylabel='accuracy test', title='Từ khoảng 50 chiều, giữ thêm gần như không được gì')
plt.tight_layout(); plt.show()
note(f"50 chiều cho accuracy {pre.iloc[3, 1]:.3f}, đủ 784 chiều cho {pre.iloc[-1, 1]:.3f}; thời gian fit {pre.iloc[3, 2]:.1f} so với {pre.iloc[-1, 2]:.1f} giây.")

## 8. Tóm tắt

In [ ]:
summary = [('số chiều giữ 90% / 95% phương sai', f'{need[0.9]} / {need[0.95]}'),
           ('sai số tái tạo với 10 / 50 chiều', f'{rec_df.iloc[1, 1]:.4f} / {rec_df.iloc[3, 1]:.4f}'),
           ('PCA: tương quan khoảng cách / giữ hàng xóm', f'{keep.iloc[0, 1]:.2f} / {keep.iloc[0, 2]:.0%}'),
           ('t-SNE 30: tương quan khoảng cách / giữ hàng xóm', f'{keep.iloc[2, 1]:.2f} / {keep.iloc[2, 2]:.0%}'),
           ('accuracy với 50 chiều / 784 chiều', f'{pre.iloc[3, 1]:.3f} / {pre.iloc[-1, 1]:.3f}')]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thay MNIST bằng Fashion-MNIST (`torchvision.datasets.FashionMNIST`) rồi xem cần bao nhiêu chiều; chạy t-SNE hai lần với `random_state` khác nhau và so hình; thêm `whiten=True` cho PCA ở mục 7.